In [0]:
%run "./Util"

In [0]:
#Create Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

#Use Catalog
spark.sql(f"USE CATALOG {CATALOG}")

#Create Bronze Schema
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
#Create Silver Schema
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
#Create Gold Schema
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

#Create Metadata Schema
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{META_SCHEMA}")

#Create Raw File Volumne
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{RAW_VOLUME}")

#Create Staging File Volumne
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{STG_VOLUME}")


DataFrame[]

In [0]:

# -- =============================================================
# -- DIM_DATE
# -- =============================================================

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} (
        date_key            INT         NOT NULL,
        full_date           DATE,
        year                INT,
        month_name          VARCHAR(50),
        month_number        INT,
        day                 VARCHAR(50),
        quarter             VARCHAR(50),
        week_number         INT,
        _date_to_warehouse  TIMESTAMP   NOT NULL,
        CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
    )
""")

# -- =============================================================
# -- DIM_RESTAURANT  (SCD Type 2)
# -- =============================================================
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT} (
        restaurant_key          BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL, 
        restaurant_id           VARCHAR(255),
        dba_name                VARCHAR(500),              
        aka_name                VARCHAR(500),                   
        facility_type           VARCHAR(255),
        address                 VARCHAR(500),
        city                    VARCHAR(50),
        state                   VARCHAR(50),
        zip_code                VARCHAR(10),
        change_hash             VARCHAR(50)     NOT NULL,
        effective_start_date    TIMESTAMP       NOT NULL,
        effective_end_date      TIMESTAMP       NOT NULL,  
        is_current              BOOLEAN         NOT NULL, 
        _date_to_warehouse      TIMESTAMP       NOT NULL,
        CONSTRAINT pk_dim_restaurant PRIMARY KEY (restaurant_key)
    )
""")

#-- =============================================================
#-- DIM_VIOLATION
#-- =============================================================
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} (
        violation_code_key      BIGINT GENERATED ALWAYS AS IDENTITY  NOT NULL,
        violation_id            VARCHAR(50)     NOT NULL, 
        violation_code          VARCHAR(50),
        violation_description   VARCHAR(1000),
        violation_detail        VARCHAR(1000),           
        _date_to_warehouse      TIMESTAMP       NOT NULL,
        CONSTRAINT pk_dim_violation PRIMARY KEY (violation_code_key)
    )
""")

# -- =============================================================
# -- FACT_INSPECTIONS
# -- =============================================================

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS}  (
        inspection_key          BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,  
        inspection_id           VARCHAR(255)    NOT NULL, 
        inspection_date_key     INT,                    
        restaurant_key          BIGINT,
        license_num             INT,
        risk_category           VARCHAR(50),                     
        inspection_score        INT,
        inspection_result       VARCHAR(100),
        total_violation         INT,
        total_violation_points  INT,
        inspection_type         VARCHAR(100),
        source_city             VARCHAR(50),          
        pass_fail_flag          VARCHAR(10),   
        _date_to_warehouse      TIMESTAMP       NOT NULL,
        CONSTRAINT pk_fact_inspections PRIMARY KEY (inspection_key),
        CONSTRAINT fk_fact_insp_date
            FOREIGN KEY (inspection_date_key)
            REFERENCES {CATALOG}.{GOLD_SCHEMA}.{DIM_DATE} (date_key),
        CONSTRAINT fk_fact_insp_restaurant
            FOREIGN KEY (restaurant_key)
            REFERENCES {CATALOG}.{GOLD_SCHEMA}.{DIM_RESTAURANT} (restaurant_key)
    )
""")

#-- =============================================================
#-- FACT_VIOLATIONS
#-- =============================================================
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{FACT_VIOLATIONS} (
        violation_key       BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,  
        inspection_key      INT,                    
        violation_code_key  INT,                
        violation_points    INT,
        is_critical         VARCHAR(10),            
        violation_comment   VARCHAR(10000),
        _date_to_warehouse  TIMESTAMP       NOT NULL,
        CONSTRAINT pk_fact_violations PRIMARY KEY (violation_key),
        CONSTRAINT fk_fact_viol_inspection
            FOREIGN KEY (inspection_key)
            REFERENCES {CATALOG}.{GOLD_SCHEMA}.{FACT_INSPECTIONS} (inspection_key),
        CONSTRAINT fk_fact_viol_violation
            FOREIGN KEY (violation_code_key)
            REFERENCES {CATALOG}.{GOLD_SCHEMA}.{DIM_VIOLATION_DETAIL} (violation_code_key)
    )
""")


#-- =============================================================
#-- QUARANTINE_INSPECTION
#-- =============================================================
spark.sql(f""" 
    CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.{GOLD_QUARANTINE_INSPECTION} (
        inspection_id            STRING,
        restaurant_id            STRING,
        dba_name                 STRING,
        aka_name                 STRING,
        inspection_date          DATE,
        inspection_type          STRING,
        inspection_result        STRING,
        violation_score          INT,
        violation_code           STRING,
        violation_points         INT,
        pass_fail_flag           STRING,
        source_city              STRING,
        address                  STRING,
        zip_code                 STRING,
        quarantine_category      STRING, 
        failed_column            STRING,
        source_system            STRING,
        pipeline_name            STRING,
        _date_to_warehouse       TIMESTAMP,
        quarantine_timestamp     TIMESTAMP
    )
""")

# -- =============================================================
# -- VERIFY TABLES CREATED
# -- =============================================================

spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA} ").display()

database,tableName,isTemporary
gold_zone,dim_date,false
gold_zone,dim_restaurant,false
gold_zone,dim_violation_detail,false
gold_zone,fact_inspections,false
gold_zone,fact_violations,false
gold_zone,quarantine_inspection,false


### Metadata

In [0]:
spark.sql(f""" 
    CREATE TABLE IF NOT EXISTS {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE} (
        table_name    STRING,
        file_name     STRING,
        active_flag   STRING,
        created_date  DATE,
        modified_date DATE
    )
""")

spark.sql(f""" 
    CREATE TABLE IF NOT EXISTS {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE}  (
        table_name        STRING,
        execution_time    TIMESTAMP,
        status            STRING,
        source_row_count  LONG,
        target_row_count  LONG,
        file_location     STRING,
        created_date      DATE
    )
""")     

DataFrame[]

In [0]:
spark.sql(f""" 
    INSERT INTO {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE} VALUES
    ('Dallas', 'Dallas_Restaurant_and_Food_Establishment_Inspections', 'Y', current_date(), current_date()),
    ('Chicago', 'Chicago_Restaurant_and_Food_Establishment_Inspections', 'Y', current_date(), current_date());
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]